# AI Earnings Call Analyzer
## Multi-Agent Intelligence System for Financial Analysis

**Author:** Nikhil Patel  
**Date:** April 2026  
**Stack:** Python, LangGraph, Claude API, pdfplumber, pandas, matplotlib

---

### Overview

This system takes a quarterly earnings call transcript and runs it through a **4-agent pipeline** to auto-generate a structured analyst report:

| Agent | Role | Output |
|-------|------|--------|
| **Financial Metrics Extractor** | Extracts revenue, EPS, margins, guidance, YoY changes | Structured metrics JSON |
| **Executive Sentiment Analyzer** | Analyzes management tone — confident, cautious, evasive | Sentiment scores + flags |
| **Guidance vs Reality Tracker** | Compares current results against prior quarter guidance | Accuracy scorecard |
| **Report Generator** | Synthesizes all agent outputs into a one-page brief | Investment brief with signal |

**Architecture:** LangGraph state machine with shared state, each agent is a node that enriches the state before passing to the next.

```
Transcript → [Metrics Agent] → [Sentiment Agent] → [Guidance Agent] → [Report Agent] → Analyst Brief
```

---
## Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install langgraph langchain-anthropic langchain-core pdfplumber pandas matplotlib python-dotenv

In [ ]:
import os
import json
import re
from datetime import datetime
from typing import TypedDict, Annotated, Literal

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pdfplumber

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

from dotenv import load_dotenv
load_dotenv()

# Initialize Claude
llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0,
    max_tokens=4096,
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

print("Environment loaded. Claude API ready.")

---
## Step 1: Transcript Ingestion

Load an earnings call transcript from PDF or text file. We extract the raw text and split it into sections (prepared remarks, Q&A) for targeted agent analysis.

In [ ]:
def extract_transcript(file_path: str) -> dict:
    """Extract and structure an earnings call transcript from PDF or text file."""
    
    # Read raw text
    if file_path.endswith('.pdf'):
        with pdfplumber.open(file_path) as pdf:
            raw_text = '\n'.join(page.extract_text() or '' for page in pdf.pages)
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
    
    # Split into sections
    sections = {
        'full_text': raw_text,
        'prepared_remarks': '',
        'qa_section': '',
        'word_count': len(raw_text.split()),
        'char_count': len(raw_text)
    }
    
    # Common section markers in earnings transcripts
    qa_markers = [
        r'(?i)question.{0,5}and.{0,5}answer',
        r'(?i)q\s*&\s*a\s+session',
        r'(?i)operator.*first question',
        r'(?i)we.{0,10}now.{0,10}open.{0,20}questions',
        r'(?i)let me now turn.{0,30}questions'
    ]
    
    split_pos = len(raw_text)
    for marker in qa_markers:
        match = re.search(marker, raw_text)
        if match:
            split_pos = min(split_pos, match.start())
    
    if split_pos < len(raw_text):
        sections['prepared_remarks'] = raw_text[:split_pos].strip()
        sections['qa_section'] = raw_text[split_pos:].strip()
    else:
        sections['prepared_remarks'] = raw_text
        sections['qa_section'] = ''
    
    return sections


# Load transcript
TRANSCRIPT_PATH = 'sample_transcripts/apple_q1_2025.txt'  # Change to your transcript
transcript = extract_transcript(TRANSCRIPT_PATH)

print(f"Transcript loaded: {transcript['word_count']:,} words")
print(f"Prepared remarks: {len(transcript['prepared_remarks']):,} chars")
print(f"Q&A section: {len(transcript['qa_section']):,} chars")
print(f"\nFirst 500 chars:\n{transcript['full_text'][:500]}")

---
## Step 2: Define the Agent State

LangGraph uses a shared state object that flows through all agents. Each agent reads from and writes to this state. This is the backbone of the multi-agent pipeline.

In [ ]:
class AnalystState(TypedDict):
    """Shared state flowing through the multi-agent pipeline."""
    
    # Input
    transcript: dict              # Raw transcript sections
    company_name: str             # Company being analyzed
    quarter: str                  # e.g., "Q1 2025"
    
    # Agent 1: Financial Metrics
    metrics: dict                 # Extracted financial metrics
    
    # Agent 2: Sentiment Analysis
    sentiment: dict               # Sentiment scores and flags
    
    # Agent 3: Guidance Tracking
    guidance: dict                # Guidance vs actuals comparison
    
    # Agent 4: Final Report
    report: str                   # Generated analyst brief
    signal: str                   # BUY / HOLD / WATCH
    
    # Metadata
    processing_log: list          # Track agent execution


print("State schema defined.")

---
## Step 3: Agent 1 — Financial Metrics Extractor

This agent scans the transcript for key financial metrics: revenue, EPS, operating margin, free cash flow, subscriber counts, and forward guidance. It returns structured JSON.

In [ ]:
def financial_metrics_agent(state: AnalystState) -> AnalystState:
    """Agent 1: Extract structured financial metrics from transcript."""
    
    print("Agent 1: Financial Metrics Extractor running...")
    
    # Use prepared remarks (where numbers are typically stated)
    text = state['transcript']['prepared_remarks'] or state['transcript']['full_text']
    
    # Truncate to fit context window
    text = text[:12000]
    
    prompt = f"""You are a financial analyst extracting metrics from an earnings call transcript.

Analyze this transcript for {state['company_name']} ({state['quarter']}) and extract ALL mentioned financial metrics.

Return ONLY valid JSON in this exact format (use null for any metrics not mentioned):
{{
    "revenue": {{"value": "$XX.XB", "yoy_change": "+X%", "beat_miss": "beat/miss/inline"}},
    "eps": {{"value": "$X.XX", "yoy_change": "+X%", "beat_miss": "beat/miss/inline"}},
    "gross_margin": {{"value": "XX.X%", "yoy_change": "+X.Xpp"}},
    "operating_margin": {{"value": "XX.X%", "yoy_change": "+X.Xpp"}},
    "free_cash_flow": {{"value": "$XX.XB", "yoy_change": "+X%"}},
    "net_income": {{"value": "$XX.XB", "yoy_change": "+X%"}},
    "forward_guidance": {{
        "next_quarter_revenue": "$XX-XXB",
        "full_year_revenue": "$XXB",
        "commentary": "brief summary of forward outlook"
    }},
    "key_segments": [
        {{"name": "segment name", "revenue": "$XB", "growth": "+X%"}}
    ],
    "notable_metrics": [
        "any other significant numbers mentioned (subscribers, units, etc.)"
    ]
}}

TRANSCRIPT:
{text}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a precise financial data extraction agent. Return ONLY valid JSON, no markdown."),
        HumanMessage(content=prompt)
    ])
    
    # Parse JSON from response
    try:
        # Clean response - remove markdown code blocks if present
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        metrics = json.loads(raw)
    except json.JSONDecodeError:
        metrics = {"error": "Failed to parse metrics", "raw_response": response.content[:500]}
    
    state['metrics'] = metrics
    state['processing_log'].append({
        'agent': 'Financial Metrics Extractor',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in metrics else 'error'
    })
    
    print(f"  Extracted {len(metrics.get('key_segments', []))} segments")
    print(f"  Revenue: {metrics.get('revenue', {}).get('value', 'N/A')}")
    print(f"  EPS: {metrics.get('eps', {}).get('value', 'N/A')}")
    
    return state


print("Agent 1 defined: Financial Metrics Extractor")

---
## Step 4: Agent 2 — Executive Sentiment Analyzer

This agent reads management's language for signals that go beyond the numbers. Is the CEO confident or hedging? Are they using forward-looking language or defensive framing? This is what separates good analysts from great ones.

In [ ]:
def sentiment_analysis_agent(state: AnalystState) -> AnalystState:
    """Agent 2: Analyze executive sentiment and tone."""
    
    print("Agent 2: Executive Sentiment Analyzer running...")
    
    text = state['transcript']['full_text'][:15000]
    
    prompt = f"""You are an expert at reading between the lines in corporate communications.

Analyze the tone and sentiment of this earnings call transcript for {state['company_name']} ({state['quarter']}).

Return ONLY valid JSON:
{{
    "overall_sentiment": "bullish/neutral/cautious/bearish",
    "confidence_score": 0.0-1.0,
    "ceo_tone": {{
        "descriptor": "confident/measured/defensive/evasive/enthusiastic",
        "evidence": "direct quote or paraphrase demonstrating tone"
    }},
    "cfo_tone": {{
        "descriptor": "confident/measured/defensive/evasive/enthusiastic",
        "evidence": "direct quote or paraphrase demonstrating tone"
    }},
    "red_flags": [
        {{
            "flag": "description of concerning language pattern",
            "severity": "low/medium/high",
            "quote": "relevant excerpt"
        }}
    ],
    "green_flags": [
        {{
            "flag": "description of positive signal",
            "evidence": "relevant excerpt"
        }}
    ],
    "language_patterns": {{
        "hedging_frequency": "low/medium/high",
        "forward_looking_statements": "few/moderate/many",
        "competitive_mentions": "none/few/several",
        "restructuring_language": true/false
    }},
    "qa_dynamics": {{
        "analyst_pushback": "none/mild/significant",
        "management_deflections": 0,
        "notable_exchanges": "brief description of most revealing Q&A moment"
    }}
}}

TRANSCRIPT:
{text}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a corporate communications analyst specializing in detecting subtle sentiment signals. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        sentiment = json.loads(raw)
    except json.JSONDecodeError:
        sentiment = {"error": "Failed to parse sentiment", "raw_response": response.content[:500]}
    
    state['sentiment'] = sentiment
    state['processing_log'].append({
        'agent': 'Executive Sentiment Analyzer',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in sentiment else 'error'
    })
    
    print(f"  Overall sentiment: {sentiment.get('overall_sentiment', 'N/A')}")
    print(f"  Confidence score: {sentiment.get('confidence_score', 'N/A')}")
    print(f"  Red flags: {len(sentiment.get('red_flags', []))}")
    print(f"  Green flags: {len(sentiment.get('green_flags', []))}")
    
    return state


print("Agent 2 defined: Executive Sentiment Analyzer")

---
## Step 5: Agent 3 — Guidance vs Reality Tracker

This agent compares what management promised last quarter against what they actually delivered. Track record matters — a company that consistently beats guidance signals strong execution. One that consistently misses signals either sandbagging or operational issues.

In [ ]:
def guidance_tracker_agent(state: AnalystState) -> AnalystState:
    """Agent 3: Compare guidance vs actual results."""
    
    print("Agent 3: Guidance vs Reality Tracker running...")
    
    text = state['transcript']['full_text'][:12000]
    metrics_context = json.dumps(state['metrics'], indent=2, default=str)
    
    prompt = f"""You are a financial analyst tracking management credibility by comparing guidance vs actual results.

Based on this {state['company_name']} ({state['quarter']}) earnings call transcript and extracted metrics, analyze the guidance accuracy.

Previously extracted metrics:
{metrics_context}

Return ONLY valid JSON:
{{
    "guidance_accuracy": {{
        "overall_score": "A/B/C/D/F",
        "description": "How well did management deliver on prior promises?"
    }},
    "beats": [
        {{"metric": "name", "guided": "value", "actual": "value", "delta": "+X%"}}
    ],
    "misses": [
        {{"metric": "name", "guided": "value", "actual": "value", "delta": "-X%"}}
    ],
    "new_guidance": [
        {{"metric": "name", "target": "value", "timeframe": "next quarter/full year", "confidence": "high/medium/low"}}
    ],
    "management_credibility": {{
        "score": 0.0-1.0,
        "pattern": "consistently beats/mixed/consistently misses/sandbagging",
        "commentary": "brief assessment of management's track record"
    }},
    "risk_factors": [
        "specific risks or headwinds mentioned that could impact forward guidance"
    ],
    "catalysts": [
        "specific positive drivers mentioned that could accelerate growth"
    ]
}}

TRANSCRIPT:
{text}"""
    
    response = llm.invoke([
        SystemMessage(content="You are a guidance tracking specialist. Compare promises vs delivery. Return ONLY valid JSON."),
        HumanMessage(content=prompt)
    ])
    
    try:
        raw = response.content.strip()
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        guidance = json.loads(raw)
    except json.JSONDecodeError:
        guidance = {"error": "Failed to parse guidance", "raw_response": response.content[:500]}
    
    state['guidance'] = guidance
    state['processing_log'].append({
        'agent': 'Guidance vs Reality Tracker',
        'timestamp': datetime.now().isoformat(),
        'status': 'success' if 'error' not in guidance else 'error'
    })
    
    print(f"  Guidance accuracy: {guidance.get('guidance_accuracy', {}).get('overall_score', 'N/A')}")
    print(f"  Beats: {len(guidance.get('beats', []))}")
    print(f"  Misses: {len(guidance.get('misses', []))}")
    print(f"  Risk factors: {len(guidance.get('risk_factors', []))}")
    
    return state


print("Agent 3 defined: Guidance vs Reality Tracker")

---
## Step 6: Agent 4 — Report Generator

The synthesis agent. Takes all outputs from Agents 1-3 and generates a structured one-page analyst brief with a clear BUY/HOLD/WATCH signal. This is the output that goes to the portfolio.

In [ ]:
def report_generator_agent(state: AnalystState) -> AnalystState:
    """Agent 4: Generate the final analyst report."""
    
    print("Agent 4: Report Generator running...")
    
    # Compile all agent outputs
    metrics_summary = json.dumps(state['metrics'], indent=2, default=str)
    sentiment_summary = json.dumps(state['sentiment'], indent=2, default=str)
    guidance_summary = json.dumps(state['guidance'], indent=2, default=str)
    
    prompt = f"""You are a senior equity research analyst writing a one-page investment brief.

Using the following analysis from three specialized agents, generate a concise, actionable analyst report for {state['company_name']} ({state['quarter']}).

FINANCIAL METRICS:
{metrics_summary}

EXECUTIVE SENTIMENT:
{sentiment_summary}

GUIDANCE TRACKING:
{guidance_summary}

Write the report in this EXACT markdown format:

# {state['company_name']} — {state['quarter']} Earnings Analysis

**Signal: [BUY / HOLD / WATCH]**  
**Confidence: [High / Medium / Low]**  
**Date: {datetime.now().strftime('%B %d, %Y')}**

---

## Executive Summary
[3-4 sentences capturing the quarter's story — what happened, why it matters, what's next]

## Key Metrics
[Table of most important financial metrics with YoY changes]

## Sentiment Read
[2-3 sentences on management tone, confidence level, and any red/green flags]

## Guidance Assessment  
[2-3 sentences on guidance accuracy, new targets, and credibility]

## Bull Case
[3 bullet points — strongest arguments for the stock]

## Bear Case
[3 bullet points — biggest risks and concerns]

## What to Watch Next Quarter
[3 specific metrics or events to monitor]

---
*Generated by AI Earnings Call Analyzer — Multi-Agent System by Nikhil Patel*

IMPORTANT: The signal (BUY/HOLD/WATCH) must be justified by the data. Do not default to HOLD."""
    
    response = llm.invoke([
        SystemMessage(content="You are a senior equity research analyst. Write clear, data-driven investment briefs. Be decisive on the signal."),
        HumanMessage(content=prompt)
    ])
    
    report = response.content.strip()
    
    # Extract signal from report
    signal = 'HOLD'  # default
    signal_match = re.search(r'\*\*Signal:\s*(BUY|HOLD|WATCH)\*\*', report)
    if signal_match:
        signal = signal_match.group(1)
    
    state['report'] = report
    state['signal'] = signal
    state['processing_log'].append({
        'agent': 'Report Generator',
        'timestamp': datetime.now().isoformat(),
        'status': 'success'
    })
    
    print(f"  Signal: {signal}")
    print(f"  Report length: {len(report):,} chars")
    
    return state


print("Agent 4 defined: Report Generator")

---
## Step 7: Build the LangGraph Pipeline

Wire all four agents into a LangGraph state machine. Each agent is a node, and the graph defines the execution flow. The state accumulates intelligence as it passes through each node.

In [ ]:
# Build the LangGraph
workflow = StateGraph(AnalystState)

# Add agent nodes
workflow.add_node("metrics_agent", financial_metrics_agent)
workflow.add_node("sentiment_agent", sentiment_analysis_agent)
workflow.add_node("guidance_agent", guidance_tracker_agent)
workflow.add_node("report_agent", report_generator_agent)

# Define the flow: linear pipeline
workflow.set_entry_point("metrics_agent")
workflow.add_edge("metrics_agent", "sentiment_agent")
workflow.add_edge("sentiment_agent", "guidance_agent")
workflow.add_edge("guidance_agent", "report_agent")
workflow.add_edge("report_agent", END)

# Compile the graph
app = workflow.compile()

print("LangGraph pipeline compiled.")
print("Flow: Metrics → Sentiment → Guidance → Report → END")

---
## Step 8: Run the Pipeline

Execute the full multi-agent pipeline on the loaded transcript. Watch each agent process in sequence.

In [ ]:
# Initialize state
initial_state = {
    'transcript': transcript,
    'company_name': 'Apple Inc.',       # Change to match your transcript
    'quarter': 'Q1 FY2025',             # Change to match your transcript
    'metrics': {},
    'sentiment': {},
    'guidance': {},
    'report': '',
    'signal': '',
    'processing_log': []
}

print(f"Analyzing {initial_state['company_name']} {initial_state['quarter']} earnings call...")
print("=" * 60)

# Run the pipeline
result = app.invoke(initial_state)

print("=" * 60)
print(f"\nPipeline complete. Signal: {result['signal']}")
print(f"Agents executed: {len(result['processing_log'])}")
for log in result['processing_log']:
    print(f"  [{log['status']}] {log['agent']} @ {log['timestamp']}")

---
## Step 9: Display the Analyst Report

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result['report']))

---
## Step 10: Visualizations

In [ ]:
def create_dashboard(result: dict):
    """Generate visual dashboard from pipeline results."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"{result['company_name']} {result['quarter']} — Earnings Intelligence Dashboard",
        fontsize=16, fontweight='bold', y=0.98
    )
    fig.patch.set_facecolor('#0a1128')
    
    for ax in axes.flat:
        ax.set_facecolor('#0d1635')
        ax.tick_params(colors='#a0a0a0')
        ax.spines['bottom'].set_color('#2a2a2a')
        ax.spines['left'].set_color('#2a2a2a')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Plot 1: Signal Badge
    ax1 = axes[0, 0]
    signal = result.get('signal', 'HOLD')
    signal_colors = {'BUY': '#4CC9A4', 'HOLD': '#F5C878', 'WATCH': '#e74c3c'}
    color = signal_colors.get(signal, '#F5C878')
    ax1.text(0.5, 0.6, signal, fontsize=48, fontweight='bold', color=color,
             ha='center', va='center', transform=ax1.transAxes)
    confidence = result.get('sentiment', {}).get('confidence_score', 0)
    ax1.text(0.5, 0.25, f'Confidence: {confidence:.0%}', fontsize=14,
             color='#a0a0a0', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Signal', color='white', fontsize=12, pad=10)
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # Plot 2: Segment Revenue
    ax2 = axes[0, 1]
    segments = result.get('metrics', {}).get('key_segments', [])
    if segments:
        names = [s.get('name', 'Unknown')[:15] for s in segments[:6]]
        # Extract numeric revenue values
        revenues = []
        for s in segments[:6]:
            rev_str = s.get('revenue', '0')
            rev_num = float(re.sub(r'[^\d.]', '', str(rev_str)) or 0)
            revenues.append(rev_num)
        bars = ax2.barh(names, revenues, color='#00d9ff', alpha=0.8)
        ax2.set_xlabel('Revenue ($B)', color='#a0a0a0')
        ax2.set_title('Segment Revenue', color='white', fontsize=12, pad=10)
        ax2.tick_params(axis='y', labelsize=9, colors='#a0a0a0')
    else:
        ax2.text(0.5, 0.5, 'No segment data', color='#a0a0a0',
                 ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('Segment Revenue', color='white', fontsize=12, pad=10)
    
    # Plot 3: Sentiment Breakdown
    ax3 = axes[1, 0]
    sentiment = result.get('sentiment', {})
    green_count = len(sentiment.get('green_flags', []))
    red_count = len(sentiment.get('red_flags', []))
    categories = ['Green Flags', 'Red Flags']
    counts = [green_count, red_count]
    colors_bar = ['#4CC9A4', '#e74c3c']
    ax3.bar(categories, counts, color=colors_bar, width=0.5)
    ax3.set_title('Sentiment Flags', color='white', fontsize=12, pad=10)
    ax3.tick_params(axis='x', colors='#a0a0a0')
    for i, v in enumerate(counts):
        ax3.text(i, v + 0.1, str(v), ha='center', color='white', fontweight='bold')
    
    # Plot 4: Guidance Scorecard
    ax4 = axes[1, 1]
    guidance = result.get('guidance', {})
    beats = len(guidance.get('beats', []))
    misses = len(guidance.get('misses', []))
    score = guidance.get('guidance_accuracy', {}).get('overall_score', 'N/A')
    ax4.text(0.5, 0.65, score, fontsize=48, fontweight='bold', color='#F5C878',
             ha='center', va='center', transform=ax4.transAxes)
    ax4.text(0.5, 0.30, f'{beats} beats  |  {misses} misses', fontsize=14,
             color='#a0a0a0', ha='center', va='center', transform=ax4.transAxes)
    credibility = guidance.get('management_credibility', {}).get('score', 0)
    ax4.text(0.5, 0.15, f'Credibility: {credibility:.0%}', fontsize=11,
             color='#a0a0a0', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Guidance Accuracy', color='white', fontsize=12, pad=10)
    ax4.set_xticks([])
    ax4.set_yticks([])
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('output/dashboard.png', dpi=150, bbox_inches='tight',
                facecolor='#0a1128', edgecolor='none')
    plt.show()
    print("Dashboard saved to output/dashboard.png")


create_dashboard(result)

---
## Step 11: Save Report to File

In [ ]:
# Save the analyst report as markdown
report_filename = f"output/{result['company_name'].replace(' ', '_')}_{result['quarter'].replace(' ', '_')}_report.md"
with open(report_filename, 'w') as f:
    f.write(result['report'])
print(f"Report saved to {report_filename}")

# Save full state as JSON for debugging / portfolio
state_filename = f"output/{result['company_name'].replace(' ', '_')}_{result['quarter'].replace(' ', '_')}_state.json"
serializable_state = {
    'company_name': result['company_name'],
    'quarter': result['quarter'],
    'signal': result['signal'],
    'metrics': result['metrics'],
    'sentiment': result['sentiment'],
    'guidance': result['guidance'],
    'processing_log': result['processing_log']
}
with open(state_filename, 'w') as f:
    json.dump(serializable_state, f, indent=2, default=str)
print(f"Full state saved to {state_filename}")

---
## Architecture Summary

```
                    +------------------+
                    |   Transcript     |
                    |   (PDF / Text)   |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 1:        |
                    |  Financial       |
                    |  Metrics         |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 2:        |
                    |  Executive       |
                    |  Sentiment       |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 3:        |
                    |  Guidance vs     |
                    |  Reality         |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Agent 4:        |
                    |  Report          |
                    |  Generator       |
                    +--------+---------+
                             |
                             v
                    +------------------+
                    |  Analyst Brief   |
                    |  + Dashboard     |
                    |  + Signal        |
                    +------------------+
```

### Key Design Decisions

1. **LangGraph over simple chains:** Each agent maintains typed state, enabling downstream agents to build on upstream analysis. A simple chain would lose structure.

2. **Specialized agents over one mega-prompt:** Breaking analysis into 4 focused agents produces more accurate results than a single prompt trying to do everything. Each agent has a narrow, well-defined task.

3. **JSON-structured outputs:** Every agent returns structured JSON, not free text. This enables programmatic dashboard generation and cross-agent data flow.

4. **Sentiment as a first-class signal:** Most financial analysis tools focus on numbers. The sentiment agent reads between the lines — hedging language, deflections in Q&A, competitive anxiety. These are leading indicators that numbers-only tools miss.

5. **Guidance tracking for credibility scoring:** A company's forward guidance is only valuable if management has a track record of accuracy. The guidance agent builds institutional memory of promise vs delivery.

---
*Analysis by Nikhil Patel | Part of [The Lab](https://github.com/patelnikhilc/The-Lab) — AI Product Case Studies*